# Chapter 5 Walkthrough: Treatment Patterns and Patient Journeys

This notebook replays the chapter as one executable story. Each step runs the same code printed in [`ch05_patient_journey.md`](ch05_patient_journey.md), so the outputs here match the chapter exactly. Run it from the repository root after generating the Chapter 3 data package.

The dashboard reports 3,193 Roventra line-1 entries without a washout. The VP asks how many are newly observed starts. By the end of this notebook, the 180-day washout reduces that count to 2,798 and every rule between the 2 numbers is explicit.

## 1. Build the cohorts

The journey and line cohort requires a paid launch-condition diagnosis, 180-day lookback, 90-day follow-up, and a mature study end. The initiation analysis uses the same lookback but does not require 90 future days at entry.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, "ch05_journey/scripts")
from episode_construction import build_newly_observed_cohort, load_chapter3_data

tables = load_chapter3_data(Path("ch03_data/output_data/generated_data"))
cohort, attrition = build_newly_observed_cohort(
    tables, minimum_lookback_days=180, minimum_followup_days=90
)
print(attrition.to_string(index=False))

                        stage  patients                                                       rule
Patients in source population     20000                     One row in the patient reference table
Observed qualifying diagnosis      8213 At least one encounter with ICD prefix E11.9|E11.65|E11.40
          Sufficient lookback      6562                     At least 180 covered days before index
              Analysis cohort      5637      Lookback plus at least 90 observable days after index


The lookback makes the therapy washout checkable. The 90-day future requirement is appropriate for stable journey tables, but it would select the initiation curve on future observation. The pipeline therefore writes separate cohort definitions for the 4,051-patient journey cohort and the 4,979-patient initiation cohort.

In [2]:
import subprocess
result = subprocess.run(
    ["uv", "run", "python", "ch05_journey/scripts/run_analysis.py"],
    capture_output=True, text=True,
)
print(result.stdout)

Cohort attrition:
                        stage  patients                                                       rule
Patients in source population     20000                     One row in the patient reference table
Observed qualifying diagnosis      8213 At least one encounter with ICD prefix E11.9|E11.65|E11.40
          Sufficient lookback      6562                     At least 180 covered days before index
              Analysis cohort      5637      Lookback plus at least 90 observable days after index

Initiation cohort patients: 6,393 (180-day lookback, no required future 90-day window)
Journey cohort treated patients: 3,928 of 5,637 | new to therapy after washout: 3,415 | prevalent users excluded: 513
Lines of therapy: 3,443 lines across 3,415 patients | line distribution: L1=3,415, L2=28
Roventra line-1 entries: 3,193 without washout, 2,798 with the 180-day washout (395 prevalent continuations relabeled)
Line-1 persistence: median 113 days | still on line 1 at 180 days: 19.2%


## 3. A transaction is not a therapy

Resolve status before sequencing: paid fills are exposure, rejections and reversals are access signals.

In [3]:
from episode_construction import prepare_pharmacy_events

basket = tables["products"]["product_name"].tolist()
paid, nonpaid = prepare_pharmacy_events(tables["pharmacy_claims"], cohort, basket)
events = pd.concat([paid, nonpaid], ignore_index=True)
print("post-index basket transactions by status:")
print(events.transaction_type.value_counts().to_string())

paid_ids = set(paid.patient_id)
any_ids = set(events.patient_id)
print(f"\npatients with a paid basket fill:      {len(paid_ids):>6,}")
print(f"patients with any basket transaction:  {len(any_ids):>6,}")
print(f"nonpaid only, never paid:              {len(any_ids - paid_ids):>6,}")

post-index basket transactions by status:
transaction_type
PAID        8749
PENDED      1364
REVERSED      69

patients with a paid basket fill:       3,928
patients with any basket transaction:   3,980
nonpaid only, never paid:                  52


Transaction status separates access attempts from completed treatment. The paid and nonpaid patient counts show the size of that distinction.

## 4. Lines of therapy: the worked patients

PAT00839 passes the washout, starts Nexoral, then switches to Vexpro after the Nexoral supply ends.

In [4]:
out = "ch05_journey/assets/generated_outputs"
pharmacy = tables["pharmacy_claims"]
basket = tables["products"]["product_name"].tolist()
mine = pharmacy[
    pharmacy.patient_id.eq("PAT00839") & pharmacy.product_name.isin(basket)
].sort_values("date_of_service")
print("PAT00839, diagnosis index 2024-01-26:")
print(mine[["date_of_service", "product_name", "days_supply",
            "transaction_type"]].to_string(index=False))

lines = pd.read_csv(f"{out}/lines.csv")
cols = ["line_number", "regimen", "line_start", "line_end",
        "fill_count", "entry_reason", "end_reason", "line_days"]
print("\nlines of therapy:")
print(lines.loc[lines.patient_id.eq("PAT00839"), cols].to_string(index=False))

PAT00839, diagnosis index 2024-01-26:
date_of_service product_name  days_supply transaction_type
     2024-06-20      Nexoral           30             PAID
     2024-07-22       Vexpro           30           PENDED
     2024-07-24       Vexpro           30             PAID
     2024-08-21       Vexpro           30             PAID

lines of therapy:
 line_number regimen line_start   line_end  fill_count    entry_reason   end_reason  line_days
           1 Nexoral 2024-06-20 2024-07-19           1 Initial therapy       Switch         30
           2  Vexpro 2024-07-24 2024-09-19           2          Switch Discontinued         58


PAT03874 demonstrates the addition rule: Nexoral arrives while Vexpro still has supply, after the regimen window closes, so the line advances to the combination.

In [5]:
import pandas as pd

out = "ch05_journey/assets/generated_outputs"
lines = pd.read_csv(f"{out}/lines.csv")
cols = ["line_number", "regimen", "line_start", "line_end",
        "fill_count", "entry_reason", "end_reason", "line_days"]
print(lines.loc[lines.patient_id.eq("PAT03874"), cols].to_string(index=False))

 line_number          regimen line_start   line_end  fill_count    entry_reason end_reason  line_days
           1           Vexpro 2024-07-06 2024-09-03           1 Initial therapy   Addition         60
           2 Nexoral + Vexpro 2024-08-29 2025-01-01           2        Addition   Censored        126


## 5. The cohort's lines, and Thursday's answer

Line depth, entries, ends, line-1 regimens, and the washout comparison that resolves the opening scene.

In [6]:
import pandas as pd

out = "ch05_journey/assets/generated_outputs"
lines = pd.read_csv(f"{out}/lines.csv")

print("patients by deepest line reached:",
      lines.groupby("patient_id").line_number.max().value_counts().sort_index().to_dict())
print("how lines are entered:", lines.entry_reason.value_counts().to_dict())
print("how lines end:        ", lines.end_reason.value_counts().to_dict())

print("\nline-1 regimens:")
print(pd.read_csv(f"{out}/lot_line1_summary.csv").to_string(index=False))

print("\nRoventra line entries, with and without the washout rule:")
base = pd.read_csv(f"{out}/lot_entry_shares.csv").assign(rule="180-day washout")
naive = pd.read_csv(f"{out}/lot_entry_shares_naive.csv").assign(rule="no washout")
print(pd.concat([naive, base])[["rule", "position", "line_entries", "share"]]
      .to_string(index=False))

patients by deepest line reached: {1: 3387, 2: 28}
how lines are entered: {'Initial therapy': 3415, 'Switch': 24, 'Addition': 4}
how lines end:         {'Censored': 1938, 'Discontinued': 1477, 'Switch': 24, 'Addition': 4}

line-1 regimens:
         regimen  patients  median_line_days  discontinued_share
        Roventra      2798              59.0               0.434
          Vexpro       309              67.0               0.443
         Nexoral       303              66.0               0.356
Nexoral + Vexpro         5              58.0               0.600

Roventra line entries, with and without the washout rule:
           rule position  line_entries  share
     no washout   Line 1          3193    1.0
180-day washout   Line 1          2798    1.0


Without the washout, Roventra has 3,193 line-1 entries. With it, the count is 2,798. The 395 records in between are continuing users that the no-washout view recounted as new starts.

## 6. The basket trap

Add one supportive product to the basket and watch the sequence invent clinical events.

In [7]:
import sys
sys.path.insert(0, "ch05_journey/scripts")
from pathlib import Path

from episode_construction import build_newly_observed_cohort, load_chapter3_data
from lot import construct_lines_of_therapy

tables = load_chapter3_data(Path("ch03_data/output_data/generated_data"))
cohort, _ = build_newly_observed_cohort(
    tables, minimum_lookback_days=180, minimum_followup_days=90
)
pharmacy = tables["pharmacy_claims"]

for label, basket in [
    ("market basket only      ", ["Roventra", "Nexoral", "Vexpro"]),
    ("plus the supportive med ", ["Roventra", "Nexoral", "Vexpro", "Supportive Med"]),
]:
    fills = pharmacy[
        pharmacy.transaction_type.eq("PAID")
        & pharmacy.product_name.isin(basket)
        & pharmacy.patient_id.isin(cohort.patient_id)
    ]
    lines, _ = construct_lines_of_therapy(fills, cohort)
    advanced = (lines.line_number > 1).sum()
    print(f"{label}: {len(lines):,} lines, "
          f"{advanced:,} line advances, "
          f"{lines.line_number.max()} deepest line")

market basket only      : 3,443 lines, 28 line advances, 2 deepest line


plus the supportive med : 3,757 lines, 77 line advances, 2 deepest line


## 7. Shake the rulers

One rule varies per row; the others hold at base.

In [8]:
import pandas as pd

out = "ch05_journey/assets/generated_outputs"
grid = pd.read_csv(f"{out}/lot_sensitivity.csv")
view = grid[["varied", "washout_days", "regimen_window_days", "allowable_gap_days",
             "new_to_therapy_patients", "combination_line1_share",
             "line1_discontinued_share", "roventra_line1_entry_share"]]
print(view.to_string(index=False))

        varied  washout_days  regimen_window_days  allowable_gap_days  new_to_therapy_patients  combination_line1_share  line1_discontinued_share  roventra_line1_entry_share
       washout             0                   30                  60                     3928                    0.001                     0.474                         1.0
       washout            90                   30                  60                     3444                    0.001                     0.426                         1.0
       washout           180                   30                  60                     3415                    0.001                     0.428                         1.0
regimen window           180                   14                  60                     3415                    0.000                     0.428                         1.0
regimen window           180                   45                  60                     3415                    0.004           

The allowable gap moves the discontinuation result because it changes when a refill gap becomes an event. The Roventra entry share is less sensitive in this package.

## 8. Time to treatment, with censoring

A patient remains in the risk set while the record is observable and treatment has not started. Treatment creates an event. Coverage end or the study cutoff creates right censoring, which removes the patient from later denominators. Kaplan-Meier multiplies the fraction remaining untreated at each treatment day. Its complement, `1 - S(t)`, is cumulative initiation when no competing event prevents treatment.

In [9]:
import pandas as pd

out = "ch05_journey/assets/generated_outputs"
journeys = pd.read_csv(f"{out}/initiation_journeys.csv")
treated = journeys[journeys.initiated_treatment]
print(f"naive view: {len(treated):,} treated of {len(journeys):,} cohort patients "
      f"({1 - journeys.initiated_treatment.mean():.1%} 'untreated')")
print(f"naive median days to treatment, treated only: {treated.days_to_treatment.median():.0f}")

curve = pd.read_csv(f"{out}/initiation_curve.csv")
km_median = curve.loc[curve.cumulative_initiation.ge(0.5), "day"].iloc[0]
print(f"Kaplan-Meier median time to treatment: {km_median:.0f} days")
for day in (90, 180, 270):
    row = curve.loc[curve.day.le(day)].iloc[-1]
    print(f"KM cumulative initiation by day {day}: {row.cumulative_initiation:.1%} "
          f"(95% CI {row.cumulative_initiation_lower_95:.1%} to "
          f"{row.cumulative_initiation_upper_95:.1%}; {int(row.at_risk):,} at risk)")

naive view: 4,110 treated of 6,393 cohort patients (35.7% 'untreated')
naive median days to treatment, treated only: 104
Kaplan-Meier median time to treatment: 168 days
KM cumulative initiation by day 90: 30.5% (95% CI 29.4% to 31.7%; 3,957 at risk)
KM cumulative initiation by day 180: 52.9% (95% CI 51.6% to 54.3%; 2,142 at risk)
KM cumulative initiation by day 270: 74.3% (95% CI 72.9% to 75.6%; 722 at risk)


## 9. Initial-regimen persistence and adherence

Persistence measures time from the first paid fill until departure from the initial regimen. A switch, addition, restart, or confirmed discontinuation is an event. PDC measures unique covered days divided by observable days, starting at the first fill and running for up to 365 days. This analysis requires at least 90 observable days. Index-product PDC follows the starting product; market-basket PDC follows any qualifying product.

In [10]:
import pandas as pd

out = "ch05_journey/assets/generated_outputs"
persistence = pd.read_csv(f"{out}/line1_persistence.csv")
for day in (60, 90, 113, 180):
    row = persistence.loc[persistence.day.le(day)].iloc[-1]
    print(f"day {day}: {row.survival:.1%} persistent; "
          f"{int(row.at_risk):,} at risk")

product = pd.read_csv(f"{out}/adherence_index_product.csv")
basket = pd.read_csv(f"{out}/adherence_market_basket.csv")
print(f"index-product PDC: mean {product.pdc.mean():.3f}, "
      f"median {product.pdc.median():.3f}, "
      f"{product.adherent_pdc.mean():.1%} at or above 0.80")
print(f"higher basket PDC than index-product PDC: "
      f"{basket.pdc.gt(product.pdc).sum():,} patients")

day 60: 73.0% persistent; 1,776 at risk
day 90: 60.6% persistent; 1,128 at risk
day 113: 49.9% persistent; 701 at risk
day 180: 19.2% persistent; 50 at risk
index-product PDC: mean 0.445, median 0.395, 15.6% at or above 0.80
higher basket PDC than index-product PDC: 36 patients


## 10. The post-index hub pathway

Only referrals between diagnosis index and follow-up end belong to this journey. The funnel combines conversion counts with median time from referral.

In [11]:
import pandas as pd

out = "ch05_journey/assets/generated_outputs"
print(pd.read_csv(f"{out}/sp_funnel.csv").to_string(index=False))
print()
outcomes = pd.read_csv(f"{out}/sp_abandonment_outcomes.csv")
pivot = outcomes.pivot_table(index="discontinue_reason", columns="outcome",
                             values="patients", fill_value=0).astype(int)
print(pivot.to_string())

                 stage  patients  share_of_referrals  median_days_from_referral
     Referral received      2597               1.000                        0.0
Authorization approved      1958               0.754                        5.0
               Shipped      1836               0.707                       10.0
             Abandoned       722               0.278                        4.0

outcome             Later paid Roventra fill  No further paid basket fill
discontinue_reason                                                       
Cost                                     236                            7
Coverage                                 198                            3
Documentation                            218                            5
Lost follow-up                            26                            0
Patient decision                          29                            0


## Exercises

1. **Flip the restart rule** (`restart_advances_line=False`) and explain the null.
2. **Payer journeys**: line-1 persistence by payer; verify the null and name the confounder you would check on real data.
3. **The immature tail**: move `study_end` to 2025-01-31 and measure the maturity artifact.

Worked answers with discussion: [`exercise_solutions.ipynb`](exercise_solutions.ipynb).